# Baseline — medical concept extraction (Vietnamese clinical notes)

GLiNER zero-shot NER + ConText rule assertions + SapBERT/FAISS linking. No training, runs on a free Colab **T4**.

See `docs/02_baseline.md` for the walkthrough.

## 1. Setup

Clone the `course` branch and install. Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [1]:
!git clone https://github.com/duongtruongbinh/viettel_ai_race_task2 medextract
%cd medextract
!pip -q install -r requirements.txt
!pip -q install -e .

fatal: destination path 'medextract' already exists and is not an empty directory.
/content/medextract
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for medextract (pyproject.toml) ... done


## 2. Knowledge bases

The linking step needs the RxNorm + ICD-10 knowledge bases. Their source files are license-gated, so download them yourself (see `INSTALL.md`) and place them in `data/kb/raw/`, then build the indexes.

> If you already have prebuilt `data/kb/processed/*.parquet` + `*.faiss` (e.g. from Google Drive), copy them into `data/kb/processed/` and skip the build cell.

In [2]:
# after placing the raw sources in data/kb/raw/ :
!python -m medextract.kb.build_rxnorm
!python -m medextract.kb.build_icd
!python -m medextract.kb.index --device auto

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/medextract/src/medextract/kb/build_rxnorm.py", line 107, in <module>
    main()
  File "/content/medextract/src/medextract/kb/build_rxnorm.py", line 97, in main
    df = build(Path(args.raw), Path(args.out), synonyms=args.synonyms)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/medextract/src/medextract/kb/build_rxnorm.py", line 58, in build
    df = _read_raw(raw)
         ^^^^^^^^^^^^^^
  File "/content/medextract/src/medextract/kb/build_rxnorm.py", line 53, in _read_raw
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
         

## 3. Run the baseline on the sample notes

In [3]:
!python run.py --config configs/baseline.yaml --input data/sample_input --output out/demo

2026-07-26 08:56:18,044 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-07-26 08:56:22,724 INFO medextract.kb.index: loading SapBERT cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR on cuda:0
2026-07-26 08:56:23,042 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 08:56:23,048 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/47b6bd041ba61311584bb2494edfda5c7d9b719f/config.json "HTTP/1.1 200 OK"
2026-07-26 08:56:23,287 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 08:56:23,288 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and 

## 4. Look at a prediction

In [4]:
import json
print(json.dumps(json.load(open('out/demo/10.json')), ensure_ascii=False, indent=2))

[
  {
    "text": "đái tháo đường type 2",
    "position": [
      10,
      31
    ],
    "type": "CHẨN_ĐOÁN",
    "assertions": [],
    "candidates": [
      "E11",
      "E26.1"
    ]
  },
  {
    "text": "Than phiền",
    "position": [
      50,
      60
    ],
    "type": "TRIỆU_CHỨNG",
    "assertions": [],
    "candidates": []
  },
  {
    "text": "tê bì hai bàn chân",
    "position": [
      61,
      79
    ],
    "type": "TRIỆU_CHỨNG",
    "assertions": [],
    "candidates": []
  },
  {
    "text": "tiểu nhiều",
    "position": [
      83,
      93
    ],
    "type": "TRIỆU_CHỨNG",
    "assertions": [],
    "candidates": []
  },
  {
    "text": "metformin 500 mg po bid",
    "position": [
      112,
      135
    ],
    "type": "THUỐC",
    "assertions": [],
    "candidates": [
      "861007"
    ]
  },
  {
    "text": "insulin glargine",
    "position": [
      139,
      155
    ],
    "type": "THUỐC",
    "assertions": [],
    "candidates": []
  },
  {
    "text": "9,2 mmo

## 5. Score on the dev set

In [6]:
!python run.py --config configs/baseline.yaml --input data/input --output out/dev_base
!python score.py --pred out/dev_base --gold data/dev

2026-07-26 08:58:13,918 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-07-26 08:58:18,767 INFO medextract.kb.index: loading SapBERT cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR on cuda:0
2026-07-26 08:58:19,064 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 08:58:19,069 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/47b6bd041ba61311584bb2494edfda5c7d9b719f/config.json "HTTP/1.1 200 OK"
2026-07-26 08:58:19,306 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 08:58:19,307 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and 